# One type, three semantics

> When is consensus among independent Cogs required?
>
> — [@oliphant2026, §5.6]

The manifest answers with a threshold: `consensus_disagreement > 0.25`.
But the rule does not say what the number measures. Rather than choose
for the author, the specification fixes the metric's **type** (a
fact-by-checker answer matrix of nullable Booleans, mapped to a scalar
in [0, 1] against the factored threshold) and exhibits three
type-satisfying metrics with different semantics, evaluated on the same
matrix. All three are correctly constructed. They do not agree on
whether the gate fires.

The answer's base type is a nullable Bool, chosen for type-checking
support as much as for clarity. The semantics: given a datum as
context, and irrespective of which Cog produced it, each checker
answers one question, "do you agree with the datum?". True is
agreement, False is disagreement, and null is cannot tell (the same
outcome the Track records as `earl:cantTell`). Checkers are factored
apart from generators on purpose: it is strictly possible for a Cog to
produce a datum and still answer False when asked whether that datum is
true. One could assume a producer always files True about its own
output; the specification does not (a clarification logged with GAP-04
in the appendix).

**A correctly constructed policy is not the same as a fit-for-purpose
policy.** What is right for the job here is an open algorithmic policy
design question, likely to have no unique correct answer but many
viable answers with different implications on incentives, and the right
answer is not limited to these three.

## The type, in the model

The abstract definition carries the type signature; three concrete
definitions specialize it, each documenting its semantics and its
incentive implication. The table is parsed from the model: each row is
one definition with its doc, verbatim.

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.show_metric_definitions()

Definition,"Doc, verbatim from the model"
abstract part def DisagreementMetric,"adjudicated: GAP-04 — the rule does not say what the number is, and no metric is chosen HERE. This abstract def is the type signature: a metric maps a fact-by-checker answer matrix of nullable Booleans to a scalar in [0, 1] comparable against the manifest's 0.25 threshold. Each answer is one checker's reply, given the datum as context and irrespective of which Cog produced it, to the question ""do you agree with the datum?"": True = agree, False = disagree, null = cannot tell (the outcome the Track records as earl:cantTell). Checkers are factored apart from generators: the Cog that produced a datum may itself answer False when asked whether it stands; no producer is assumed to file True (GAP-04 clarification, 2026-09-04). Three type-satisfying alternatives with different semantics follow; which is fit for purpose is an open algorithmic policy design question, unlikely to have a unique correct answer, with different implications on incentives. The right answer is not limited to these three."
part def DissentFractionMetric :> DisagreementMetric,"Fraction of answers that are not True: disagreement and null (cannot tell) both count as dissent. Incentive implication: honest abstention is punished, so checkers are pushed to affirm rather than say cannot tell. Note: on the paper's own section 5.5 example (three Cogs, two must agree) this metric FIRES the gate at the 0.25 threshold, contradicting the example's intent."
part def StrictQuorumUndecodableMetric :> DisagreementMetric,"Fraction of facts where no answer value reaches a quorum of two identical non-null answers; null blocks quorum. Incentive implication: abstention is conservative: it escalates more runs to expert review, spending review capacity. Consistent with the section 5.5 example."
part def ErasureAwareUndecodableMetric :> DisagreementMetric,"Fraction of facts undecodable among non-null answers only: null (cannot tell) is an erasure, not an error, so a lone expressed voice among abstentions decodes. Incentive implication: abstention shifts power to whoever still answers. Consistent with the section 5.5 example."


The structural punchline sits in the comparator oracle. Its metric
slot is typed by the abstract definition and deliberately left unbound
(`part metric : DisagreementMetric;` with no concrete binding): the
model states that a metric of this type is required and refuses to
choose one (GAP-04 in the appendix, adjudicated open). A deployment
binds it; the Track records which one a run used.

In [2]:
exhibits.show_unbound_metric_slot()

part def ConsensusComparatorOracle :> OracleService {
    doc /* adjudicated: GAP-04 — the metric slot below is typed by
       the abstract DisagreementMetric and deliberately left
       unbound: the model states that a metric of this type is
       required and refuses to choose one. A deployment binds it;
       the Track records which one a run used. */
    part metric : DisagreementMetric;
    attribute returnedDisagreement : ScalarValues::Real;
    port reading : ReadingWrite;
}


## The three implementations

The same three semantics as running code, satisfying the declared type:

- **dissent-fraction** counts every answer that is not True, null
  included: honest abstention is punished, so checkers are pushed to
  affirm rather than say cannot tell. On the paper's own section 5.5
  example (three Cogs, two must agree) it FIRES the gate at 0.25,
  contradicting the example's intent.
- **strict-quorum-undecodable** counts facts where no answer reaches a
  quorum of two matching non-null answers; null blocks quorum, so it
  escalates more, spending expert-review capacity.
- **erasure-aware-undecodable** treats null as an erasure, not an
  error: a lone expressed voice among abstentions decodes, so
  abstention shifts power to whoever still answers.

In [3]:
import inspect
for f in (exhibits.dissent_fraction, exhibits.strict_quorum_undecodable,
          exhibits.erasure_aware_undecodable):
    print(inspect.getsource(f))

def dissent_fraction(matrix):
    dissents = total = 0
    for fact in matrix:
        dissents += sum(1 for v in fact if v is not True)
        total += len(fact)
    return dissents / total

def strict_quorum_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v is not None]
        if not expressed or max(expressed.count(v) for v in set(expressed)) < 2:
            undecodable += 1
    return undecodable / len(matrix)

def erasure_aware_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v is not None]
        if not expressed or max(expressed.count(v) for v in set(expressed)) * 2 <= len(expressed):
            undecodable += 1
    return undecodable / len(matrix)



## Where the definitions meet the implementations

The model cannot hold Python, and this walkthrough does not pretend it
does. The binding between a part def and the function that implements
its logic is a stated, mechanical convention, checked in both
directions. A definition's metric id is derived from its name
(`DissentFractionMetric` becomes `dissent-fraction`); the same id keys
the implementation mapping in `exhibits.py` and is the value a Track
records in `vfr:metric`. The cell verifies the correspondence is
one-to-one — every model definition has an implementation, every
implementation has a definition — and that the metric run-001 cited
resolves to a bound implementation. In a deployment, the same id is how
the comparator oracle's configuration names the metric it bound into
the open slot.

In [4]:
exhibits.show_metric_binding()

Model definition,Metric id (Track vocabulary),Implementation,Bound
DissentFractionMetric,dissent-fraction,exhibits.dissent_fraction,✓
StrictQuorumUndecodableMetric,strict-quorum-undecodable,exhibits.strict_quorum_undecodable,✓
ErasureAwareUndecodableMetric,erasure-aware-undecodable,exhibits.erasure_aware_undecodable,✓


model <-> implementation binding: OK (3 definitions, 3 implementations)
run-001 recorded metric: strict-quorum-undecodable -> bound implementation: exhibits.strict_quorum_undecodable


## One matrix, three answers

The same answer matrix, the threshold read from the model's single
point of definition, and three different gate decisions. The Track
records which metric a run actually used; recording is not
endorsing.

In [5]:
exhibits.evaluate_metrics()

type signature: DisagreementMetric = (fact x checker answer matrix of nullable Bool; True = agree, False = disagree, null = cannot tell) -> [0, 1]

same answer matrix for all three metrics (8 facts x 3 checkers):
  f1: True  True  True 
  f2: True  True  True 
  f3: True  True  True 
  f4: True  True  False
  f5: True  True  null 
  f6: True  False null 
  f7: False null  True 
  f8: True  null  null 
threshold (from the model, single point of definition): > 0.25

dissent-fraction          : 0.333 -> gate fires: True
strict-quorum-undecodable : 0.375 -> gate fires: True
erasure-aware-undecodable : 0.250 -> gate fires: False
